In [ ]:
# prepare
import emap
import json

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    elif type_.startswith("$"): # other types
        return len(ports[0]) * 1.0
    return 0.0  # blackboxes or tech cells

### Example: Multiplier with Synchronous Reset

In [ ]:
!yosys -q -p "read_verilog tests/multiplier_with_rst.v; proc; write_json multiplier_with_rst.json"

A DSP configuration that has synchronous reset / enable / dynamic control signals is not supported by `Lakeroad` currently. However, you can still provide `Nextmap` with manual DSP mapping rules to handle these features.

In [ ]:
dsp_rules = {
    "signed_mul_2_stage_26_17_48_bit_rst": {    # rule name
        "requirements": {                       # resource requirements
            "dsp48e2": 1                        # use one DSP48E2
        },
        "hidden_inputs": ["clk"],               # hidden input ports, e.g., clock
        "inputs": ["a", "b", "rst"],            # input ports
        "outputs": ["p"],                       # output ports
        # and a match pattern in SQL
        "match_sql": """
            SELECT sdff_a.d, sdff_b.d, sdff_p.rst, sdff_p.q
            FROM sdffs AS sdff_a JOIN sdffs AS sdff_b JOIN aby_cells AS mul JOIN sdffs AS sdff_p
            ON sdff_a.q = mul.a AND sdff_b.q = mul.b AND mul.y = sdff_p.d
            WHERE mul.type = '$muls'
                AND width_of(sdff_a.d) <= 26 AND width_of(sdff_b.d) <= 17 AND width_of(sdff_p.q) <= 48
                AND sdff_a.rst = sdff_b.rst AND sdff_b.rst = sdff_p.rst
                AND sdff_a.rst_val = 0 AND sdff_b.rst_val = 0 AND sdff_p.rst_val = 0
        """
        # note that DSP48E2 can only support reset to zero
    }
}

In [ ]:
TEST_NAME = "multiplier_with_rst"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist)   # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 1}, OutputFlag=False)
with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)